# ==========================================
# EJERCICIO 1: MUESTREO PRESA (FIFA 20)
# ==========================================

In [2]:
import numpy as np
import csv

In [3]:
class MuestreoPresaFifa:
    def __init__(self, k):
        self.k = k
        self.i = 0
        # MODIFICACIÓN 1: Usamos una lista de Python para guardar texto en lugar de np.zeros
        self.presa = [None] * self.k

    def __call__(self, x):
        if self.i < self.k:
            self.presa[self.i] = x
        else:
            # MODIFICACIÓN 2: Corrección matemática sutil. 
            # randint(0, self.i + 1) asegura que la probabilidad de reemplazo sea exactamente k / (i+1)
            j = np.random.randint(0, self.i + 1)
            if j < self.k:
                self.presa[j] = x    
        self.i += 1

    def obtener_presa(self):
        return self.presa

In [4]:
# 1. Definimos cuántos jugadores queremos en nuestra muestra final
k = 10
mp_fifa = MuestreoPresaFifa(k)

print(f"Iniciando el escaneo del flujo de datos de FIFA... Buscando {k} jugadores aleatorios.")

# 2. Abrimos el archivo CSV y lo leemos línea por línea (Simulación de Data Stream)
with open('players_20.csv', 'r', encoding='utf-8') as archivo:
    lector = csv.DictReader(archivo) # Esto lee cada línea como un diccionario
    
    for fila in lector:
        # 3. Extraemos los datos que nos interesan de ese jugador (Nombre y Club)
        # Usamos .get() por seguridad en caso de que el nombre de la columna varíe
        nombre = fila.get('short_name', 'Desconocido')
        equipo = fila.get('club', 'Sin Equipo')
        
        jugador_formateado = f"{nombre} ({equipo})"
        
        # 4. Enviamos al jugador a intentar entrar a la presa
        mp_fifa(jugador_formateado)

Iniciando el escaneo del flujo de datos de FIFA... Buscando 10 jugadores aleatorios.


In [5]:
# 5. Imprimimos el resultado final
print("\n--- MUESTRA FINAL (RESERVOIR SAMPLING) ---")
muestra_final = mp_fifa.obtener_presa()

for indice, jugador in enumerate(muestra_final):
    print(f"{indice + 1}. {jugador}")


--- MUESTRA FINAL (RESERVOIR SAMPLING) ---
1. I. Sangaré (Toulouse Football Club)
2. D. Potter (Tranmere Rovers)
3. A. Lotti (Atlético Tucumán)
4. Édundo Oliveira (Cruzeiro)
5. A. Dabo (FC Nantes)
6. L. Bittencourt (TSG 1899 Hoffenheim)
7. D. Wiebe (Eintracht Braunschweig)
8. Sim Sang Min (Pohang Steelers)
9. S. Nakatani (Nagoya Grampus)
10. N. Ependi (Beijing Renhe FC)


### Análisis: Muestreo de Presa (Reservoir Sampling) en Datos Reales (FIFA)

Para adaptar nuestro algoritmo matemático a un entorno de **Big Data** real, realizamos modificaciones clave tanto en la estructura de almacenamiento como en la estrategia de lectura de datos:

**1. Rediseño del Contenedor (Manejo de Texto)**
* **El Cambio:** El arreglo original de `numpy` (`np.zeros`) está optimizado exclusivamente para números. Lo reemplazamos por una lista nativa de Python inicializada con valores nulos (`self.presa = [None] * self.k`). Esto nos permite almacenar objetos complejos, como cadenas de texto (los nombres y clubes de los jugadores).
* **Ajuste Matemático:** Se ajustó el generador aleatorio a `np.random.randint(0, self.i + 1)` para asegurar que la probabilidad de que un nuevo jugador entre a la presa sea estadísticamente perfecta ($k / i$).

**2. Procesamiento en Flujo Continuo (Simulación de Data Stream)**
* **La Trampa de la RAM:** En Big Data, usar librerías clásicas como Pandas (`pd.read_csv`) es peligroso porque intentan cargar la totalidad del archivo en la memoria RAM, lo cual colapsa el servidor si el dataset pesa Terabytes.
* **La Solución:** Utilizamos `csv.DictReader` dentro de un bloque `with open(...)`. Esta técnica actúa como un iterador perezoso (*lazy iterator*): abre el archivo, extrae la fila actual, la evalúa en la presa, y la borra de la memoria antes de leer la siguiente línea. 

**3. Muestreo en Tiempo Real**
* Conforme el archivo fluye línea a línea, extraemos las variables de interés (`short_name` y `club`) usando el método seguro `.get()`.
* El objeto `mp_fifa` actúa como un embudo. Decide en microsegundos si el jugador actual entra a la lista dorada o es descartado para siempre.

**Conclusión**
El algoritmo logró evaluar el universo completo de más de 18,000 jugadores del archivo de FIFA, pero **nuestra memoria RAM solo almacenó 10 jugadores en todo momento**. Obtuvimos una muestra aleatoria estadísticamente perfecta optimizando la complejidad espacial a $\mathcal{O}(k)$ en lugar de $\mathcal{O}(n)$.

# ==========================================
# EJERCICIO 2: MUESTREO DE BERNOULLI (FIFA 20)
# ==========================================

In [6]:
import random
import csv

In [7]:
class MuestreoBernoulliFifa:
    def __init__(self, probabilidad):
        self.p = probabilidad
        self.muestra = []
        self.total_evaluados = 0

    def __call__(self, x):
        self.total_evaluados += 1
        # Generamos un número aleatorio entre 0.0 y 1.0
        # Si cae por debajo de nuestra probabilidad 'p', el jugador entra.
        if random.random() <= self.p:
            self.muestra.append(x)

    def obtener_muestra(self):
        return self.muestra
    
    def obtener_estadisticas(self):
        return self.total_evaluados, len(self.muestra)

In [8]:
# 1. Definimos la probabilidad (ej. 0.001 = 0.1% de los datos)
probabilidad_p = 0.001
mb_fifa = MuestreoBernoulliFifa(probabilidad_p)

print(f"Iniciando escaneo... Lanzando una moneda con {probabilidad_p*100}% de probabilidad por cada jugador.")

Iniciando escaneo... Lanzando una moneda con 0.1% de probabilidad por cada jugador.


In [9]:
# 2. Leemos el archivo en flujo continuo
with open('players_20.csv', 'r', encoding='utf-8') as archivo:
    lector = csv.DictReader(archivo) 
    
    for fila in lector:
        nombre = fila.get('short_name', 'Desconocido')
        equipo = fila.get('club', 'Sin Equipo')
        jugador_formateado = f"{nombre} ({equipo})"
        
        # 3. Evaluamos al jugador
        mb_fifa(jugador_formateado)

In [10]:
# 4. Imprimimos resultados
total, capturados = mb_fifa.obtener_estadisticas()
muestra_final = mb_fifa.obtener_muestra()

print("\n--- MUESTRA FINAL (BERNOULLI SAMPLING) ---")
print(f"Jugadores evaluados en total: {total}")
print(f"Jugadores capturados en la muestra: {capturados}")
print("-" * 40)

for indice, jugador in enumerate(muestra_final):
    print(f"{indice + 1}. {jugador}")


--- MUESTRA FINAL (BERNOULLI SAMPLING) ---
Jugadores evaluados en total: 18278
Jugadores capturados en la muestra: 18
----------------------------------------
1. K. Havertz (Bayer 04 Leverkusen)
2. M. Niakhaté (1. FSV Mainz 05)
3. A. González (Estudiantes de La Plata)
4. J. Sorbon (En Avant de Guingamp)
5. Abdón Prats (RCD Mallorca)
6. S. Kingsley (Hull City)
7. B. Tuiloma (Portland Timbers)
8. P. McEleney (Dundalk)
9. C. Lobos (Universidad Católica)
10. G. Gómez (Racing Club)
11. N. Rajsel (KV Oostende)
12. N. Manfredini (Benevento)
13. M. Fayez (Al Ain FC)
14. Y. Aygün (Göztepe SK)
15. J. Toro (CD Tondela)
16. C. Daniel (Burton Albion)
17. F. Egerer (SV Meppen)
18. O. Björtuft (Odds BK)


### Análisis: Implementación de Muestreo de Bernoulli

Para complementar el algoritmo de Presa (Reservoir), se implementó el **Muestreo de Bernoulli**. Este es un algoritmo de selección probabilística independiente, altamente utilizado en motores de Big Data como Apache Spark (ej. método `.sample()`).

**Diferencias Clave vs. Reservoir Sampling:**
1. **Decisión Independiente:** En lugar de mantener una probabilidad dinámica basada en el tiempo histórico ($k/i$), Bernoulli evalúa a cada jugador de manera completamente aislada lanzando una "moneda sesgada" con una probabilidad fija $p$.
2. **Tamaño de Muestra Variable:** No existe un tamaño $k$ estricto. El tamaño final de la muestra es una variable aleatoria que sigue una Distribución Binomial con media $n \times p$. Si ejecutamos la celda varias veces, veremos que la cantidad de jugadores seleccionados fluctúa ligeramente (por ejemplo, si evaluamos 18,000 jugadores con $p=0.001$, esperamos capturar a ~18 jugadores, pero podríamos obtener 15, 19 o 22 en distintas ejecuciones).
3. **Gestión de Memoria:** Aunque procesa el flujo en tiempo real (evitando colapsar la RAM con la carga del CSV), la lista `self.muestra` crece dinámicamente. Es ideal para extraer fracciones (porcentajes) de flujos infinitos donde no nos importa el número exacto de elementos capturados, siempre y cuando la muestra sea estadísticamente representativa.